# **1. LSTM**

LSTM(Long Short-Term Memory)은 RNN의 장기 의존성 문제를 해결하기 위해 고안된 모델입니다. 
<br>LSTM은 셀 상태(cell state)와 3개의 게이트(입력 게이트, 출력 게이트, 망각 게이트)를 사용하여 
<br>중요한 정보를 오랫동안 저장하고 불필요한 정보를 제거하는 구조를 갖추고 있습니다. 
<br>망각 게이트는 이전 셀 상태에서 필요 없는 정보를 삭제하고, 입력 게이트는 새로운 정보를 저장하며, 
<br>출력 게이트는 최종 출력을 결정합니다. 이러한 구조 덕분에 LSTM은 장기 시퀀스를 다루는 
<br>자연어 처리, 음성 인식, 시계열 예측 등의 다양한 분야에서 효과적으로 사용됩니다. 
<br>하지만 구조가 복잡하여 계산량이 많고, 학습 시간이 오래 걸린다는 단점이 있습니다.

![RNN과  LSTM](./images/RNN과%20%20LSTM.png)

### RNN의 문제점
- 기본 RNN에는 주로 Hidden State(h) 하나가 기억 역할을 함
- RNN은 시간 순서대로 같은 가중치를 반복해서 사용

```
    x1 > RNN > h1 > RNN > h2 > RNN > h3 ... > ht
```
- 학습할 때는 마지막 오차를 과거 시점까지 거꾸로 전달
```
    0.5
    0.5 * 0.5 = 0.25
    0.5 * 0.5 *0.5 = 0.125
    ...
    0.5^20 = ?
```
- 멀리 갈수록 Gradient가 매우 작아짐. 그러면 오래전 입력이 현재 오차에 어떤 영향을 주었는지 가중치가 거의 학습하지 못함 > 기울기 소실
- 1보다 큰 값이 반복적으로 곱해지면 Gradient가 지나치게 커질 수 있음 > 기울기 폭주

> Gradient가 정확히 0이어야만 기울기 소실은 아님. 0에 매우 가까워져 학습에 실질적인 영향을 거의 주지 못하는 상태도 기울기 소실이라고 함

### LSTM의 핵심 구조
- LSTM 한 시점 t에는 아래와 같은 정보가 들어옴
    - x_t: 현재 시점 입력
    - h_(t-1): 이전 시점 Hidden State. 현재 시점에서 다음 시점과 출력 계산에 직접 사용되는 내부 표현
    - c_(t-1): 이전 시점 Cell State. 시간축을 따라 정보를 전달하는 주된 내부 상태. 중요한 정보가 여러 시점 동안 유지될 수 있는 경로

1. Forget Gate
- 과거 기억을 남김
- sigmoid 결과로 단순한 삭제/유지가 아니라 이전 Cell State의 각 성분을 다음 단계로 얼마나 전달할지를 나타내는 0~1 사이의 비율
```
현재 입력 x_t
                    가중치 W_f와 계산 > b_f > sigmoid > f_t
이전 상태 h_(t-1)
처음에는 W_f, b_f가 적절한 값을 모름 > 학습 과정에서 예측 > 실제 정답과 비교 > Loss 계산 > 역전파 > W_f, b_f 수정 > 다음 Gate 출력도 달라짐(반복)
```
- 어떤 상황에서 이전 정보를 많이 통과시키는 것이 Loss를 줄이는지를 모델이 학습하게 됨
- Gate의 값은 사람이 만든 규칙이 아니라 학습된 가중치와 현재 입력/이전 상ㅌ태가 함께 만들어낸 결과

2. Input Gate
- 새 정보를 얼마나 저장할지
- 새로운 후보 기억도 만듦

3. Output Gate
- 출력으로 무엇을 보여줄지
```
    이전 Cell State > Forget Gate
                                    > 원소별 곱 > 새로운 Cell State(c_t) > Output Gate > Hidden State(h_t)
    현재 입력 > Input Gate
```

> Cell State의 갱신(과거 기억 + 새 기억)은 과거 기억 중 남길 부분에서 현재 입력에서 새로 저장할 부분을 더한 값

### 장기/단기 기억을 나누는 기준

```
현재 시점  t
    - 1시간 전 값
    - 2시간 전 값
    - 3시간 전 값
    - ...
    - 24시간 전 값
```
- LSTM 내부에 6시간보다 오래되면 장기 정보, 3시간 이내면 단기 정보와 같은 규칙은 없음
- 얼마나 오래된 정보인가보다 그 정보가 예측에 얼마나 도움이 되었는가가 중요
- 예) 전력 데이터에서 "매일 저녁 비슷한 시간에 사용량이 증가" 패턴이 반복된다면, 비교적 이전 시점에서 만들어진 정보라도 다음 값을 예측하는 데 유용할 수 있음 > 장기 정보
- 예) 1시간 전 정보라도 잡음에 가까워 예측에 도움이 되지 않는 성분은 > 단기 정보

> 사람이 장기/단기 정보를 미리 지정하는 모델이 아니라 학습을 통해 어떤 과거 정보의 영향력을 계속 유지하고, 어떤 정보의 영향력을 줄일지를 결정

데이터셋: https://archive.ics.uci.edu/dataset/235/individual%2Bhousehold%2Belectric%2Bpower%2Bconsumption